### SQL
A SQLite database with tables corresponding to each table in the [MBTA GTFS Documenation](https://github.com/mbta/gtfs-documentation/blob/master/reference/gtfs.md) has been loaded for your convenience. Data exploration will be performed in SQL throughout this guide. 

In [1]:
.open feed.db

You can now execute SQL queries on the database.

In [2]:
-- Select this cell and press Shift + Return

SELECT *
FROM
	feed_info;

feed_publisher_name,feed_publisher_url,feed_lang,feed_start_date,feed_end_date,feed_version,feed_contact_email,feed_id
MBTA,http://www.mbta.com,EN,20240127,20240406,"Winter 2024, 2024-02-03T02:07:36+00:00, version D",developer@mbta.com,mbta-ma-us


## Alter `stops` to include street names as individual columns

In [6]:
ALTER TABLE
	stops
	DROP COLUMN
		stop_street;

SQLITE_ERROR: no such column: "stop_street"

In [ ]:
ALTER TABLE
	stops
	ADD COLUMN
		stop_street
		TEXT
		GENERATED
		ALWAYS
		AS
			(CASE
				WHEN instr(stop_name, ' @ ') > 0
					THEN substr(stop_name, 1, instr(stop_name, ' @ ')-1)
				ELSE NULL
			END)
		VIRTUAL;


In [ ]:
ALTER TABLE
	stops
	DROP COLUMN
		stop_cross_street;

In [ ]:
ALTER TABLE
	stops
	ADD COLUMN
		stop_cross_street
		TEXT
		GENERATED
		ALWAYS
		AS
			(CASE
				WHEN instr(stop_name, ' @ ')=0
					THEN NULL
				WHEN instr(stop_name, ' @ ') > 0
					THEN substr(stop_name, instr(stop_name, ' @ ') + 3)
				ELSE 0
			END)
		VIRTUAL;

## Finding Shapes

### Iteration 1

In [ ]:
SELECT *
FROM
	route_patterns
WHERE
	route_pattern_id
		= '71-5-1';

In [ ]:
SELECT *
FROM
	trips

WHERE
	trip_id
		= '60143923'

In [ ]:
select shape_pt_lon, shape_pt_lat
from shapes
where shape_id = '710108'
order by CAST(shape_pt_sequence as INTEGER)

In [ ]:
select FORMAT('{%s, %s},', shape_pt_lon, shape_pt_lat)
from shapes
where shape_id = '710108'
order by CAST(shape_pt_sequence as INTEGER)

In [ ]:
SELECT
	FORMAT('%%Schedule.Gtfs.Stop{ id: "%s", name: "%s", latitude: %s, longitude: %s },', stop.stop_id, stop.stop_name, stop.stop_lat, stop.stop_lon )
FROM
	stop_times
		AS stop_time
JOIN
	stops
		AS stop
	ON
		stop_time.stop_id
			= stop.stop_id
WHERE
	trip_id
		= '60143923'
ORDER BY
	stop_sequence;

### Iteration 2

Find the representative trip id and shape id's for a specific route

In [ ]:
SELECT
	pattern.route_pattern_id,
	pattern.route_pattern_name,
	pattern.representative_trip_id
		AS trip_id,

	trip.shape_id



	-- pattern.*
FROM
	route_patterns
		AS pattern
		
JOIN
	trips
		AS trip
	ON
		trip.trip_id = pattern.representative_trip_id


WHERE
	pattern.route_id = '71'


Get the stops for the trip_id

In [7]:
SELECT
	FORMAT('%%Schedule.Gtfs.Stop{ id: "%s", name: "%s", latitude: %s, longitude: %s },', stop.stop_id, stop.stop_name, stop.stop_lat, stop.stop_lon )
FROM
	stop_times
		AS stop_time
JOIN
	stops
		AS stop
	ON
		stop_time.stop_id
			= stop.stop_id
WHERE
	stop_time.trip_id
		= '60487527'
ORDER BY
	stop_sequence;

"FORMAT('%%Schedule.Gtfs.Stop{ id: ""%s"", name: ""%s"", latitude: %s, longitude: %s },', stop.stop_id, stop.stop_name, stop.stop_lat, stop.stop_lon )"
"%Schedule.Gtfs.Stop{ id: ""334"", name: ""Ashmont"", latitude: 42.284195, longitude: -71.063879 },"
"%Schedule.Gtfs.Stop{ id: ""536"", name: ""Dorchester Ave @ Hurlcroft Ave"", latitude: 42.282077, longitude: -71.065378 },"
"%Schedule.Gtfs.Stop{ id: ""537"", name: ""Dorchester Ave @ Gallivan Blvd"", latitude: 42.280386, longitude: -71.065900 },"
"%Schedule.Gtfs.Stop{ id: ""569"", name: ""Dorchester Ave @ Valley Rd"", latitude: 42.278091, longitude: -71.066507 },"
"%Schedule.Gtfs.Stop{ id: ""10569"", name: ""2165 Dorchester Ave"", latitude: 42.276285, longitude: -71.067029 },"
"%Schedule.Gtfs.Stop{ id: ""570"", name: ""Dorchester Ave @ St Gregory St"", latitude: 42.275594, longitude: -71.067229 },"
"%Schedule.Gtfs.Stop{ id: ""883"", name: ""Dorchester Ave @ Richmond St"", latitude: 42.273982, longitude: -71.067662 },"
"%Schedule.Gtfs.Stop{ id: ""571"", name: ""Dorchester Ave @ Washington St"", latitude: 42.272456, longitude: -71.068104 },"
"%Schedule.Gtfs.Stop{ id: ""3477"", name: ""Adams St @ Eliot St"", latitude: 42.270445, longitude: -71.068002 },"
"%Schedule.Gtfs.Stop{ id: ""3478"", name: ""Adams St @ Canton Ave"", latitude: 42.268924, longitude: -71.067430 },"


Get the shape points for the shape id

In [1]:
SELECT
	FORMAT('{%s, %s},', shape_pt_lon, shape_pt_lat)
FROM
	shapes
WHERE
	shape_id = '2170118'
ORDER BY
	CAST(shape_pt_sequence as INTEGER)

SQLITE_ERROR: no such table: shapes

In [22]:
SELECT DISTINCT
	count(RP.representative_trip_id)
FROM
	stop_times
		AS ST
JOIN
	trips
		AS T
	ON T.trip_id = ST.trip_id,
	route_patterns
		AS RP
	ON RP.route_pattern_id = T.route_pattern_id
WHERE
	ST.arrival_time > '24:00'
ORDER BY
	ST.arrival_time
LIMIT 10


count(RP.representative_trip_id)
59446


In [35]:
SELECT DISTINCT
	RP.route_id, RP.route_pattern_name
FROM
	route_patterns
		AS RP
WHERE
	RP.route_pattern_id IN (
		SELECT DISTINCT
			T.route_pattern_id
		FROM
			stop_times
				AS ST
		JOIN
			trips
				AS T
			ON T.trip_id = ST.trip_id
		WHERE
			ST.arrival_time > '25:00'
	)
LIMIT 10


route_id,route_pattern_name
Red,Alewife - Braintree
Red,Alewife - Ashmont
Red,Park Street - Braintree
Red,Park Street - Ashmont
Red,Braintree - Alewife
Red,Ashmont - Alewife
Mattapan,Ashmont - Mattapan
Mattapan,Mattapan - Ashmont
Orange,Oak Grove - Forest Hills
Orange,Forest Hills - Oak Grove


In [3]:

SELECT DISTINCT
	T.route_pattern_id
FROM
	stop_times
		AS ST
JOIN
	trips
		AS T
	ON T.trip_id = ST.trip_id
WHERE
	ST.arrival_time > '24:00'
LIMIT 20


route_pattern_id
Mattapan-_-0
Mattapan-_-1
Orange-3-0
Orange-3-1
Blue-6-0
Blue-6-1
741-_-1
741-_-0
741-2-1
28-_-0


In [7]:
select *
from calendar_dates
limit 10

service_id,date,exception_type,holiday_name
RTL12024-hms14011-Weekday-01,20240129,1,
RTL12024-hms14011-Weekday-01,20240130,1,
RTL12024-hms14011-Weekday-01,20240131,1,
RTL12024-hms14011-Weekday-01,20240201,1,
RTL12024-hms14011-Weekday-01,20240202,1,
FALL 2023-NORTHSAT-Saturday-12,20240203,2,
FALL 2023-NORTHSUN-Sunday-12,20240204,2,
FALL 2023-NORTHSAT-Saturday-1-S7d8482ee,20240210,2,
RTL12024-hms14016-Saturday-01,20240210,2,
FALL 2023-NORTHSUN-Sunday-1-S7d8482ee,20240211,2,
